Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Winter Semester 2025/26 (Master Course #24512)

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# PyTorch Model for Binary Logistic Regression with One Sigmoid Layer
- PyTorch is used to train the model and to make predictions
- data synthesis and data split is done with `binary_log_reg_toy_data()` which uses scikit-learn 
- statistical measures are calculated with scikit-learn
- the implementation uses 64-Bit double precision
- manual initialisation of model weights
- no shuffling of batch data -> vanilla batch gradient descent
- static learning rate
- hence, the training is **fully deterministic** and thus all results are precisely identical with those from other exemplary implementations
    - [binary_logistic_regression_manual.ipynb](binary_logistic_regression_manual.ipynb)
    - [binary_logistic_regression_tensorflow.ipynb](binary_logistic_regression_tensorflow.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import torch

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import balanced_accuracy_score, accuracy_score

from torch.utils.data import TensorDataset, DataLoader

from util_binary_logistic_regression import toy_data, init_weights
from util_binary_logistic_regression import my_sigmoid

torch.set_default_dtype(torch.float64)

# last manual check with ('2.8.0', '1.7.2')
torch.__version__, sklearn.__version__

## Data

In [ ]:
X_train, Y_train, X_test, Y_test = toy_data()
N_Features = X_train.shape[1]

## Learning Parameters

In [ ]:
batch_size = X_train.shape[0] // 400
num_epochs = 10
learning_rate = 0.1

## Prepare Data for Torch

In [ ]:
data_train = TensorDataset(
    torch.DoubleTensor(X_train), torch.DoubleTensor(Y_train))
data_test = TensorDataset(
    torch.DoubleTensor(X_test), torch.DoubleTensor(Y_test))
data_train_loader = DataLoader(
    dataset=data_train, batch_size=batch_size, shuffle=False)

## Define Torch Model

In [ ]:
class Model(torch.nn.Module):

    def __init__(self, input_size):
        super(Model, self).__init__()

        self.linear1 = torch.nn.Linear(input_size, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.linear1(x)
        x = self.sigmoid(x)
        return x

    def predict_class(self, x):
        pred = self.forward(x)
        return (pred >= 0.5).float()


model = Model(input_size=N_Features)
model

## Define Loss Function and Optimizer Strategy

In [ ]:
loss_fcn = torch.nn.BCELoss(reduction='mean')  # i.e. empirical risk
optimizer = torch.optim.SGD(model.parameters(),
                            lr=learning_rate)

## Init Model Parameters
to obtain reproducible results with the other implementations

In [ ]:
w1, w2, b = init_weights()
with torch.no_grad():
    model.linear1.weight[0, 0] = w1
    model.linear1.weight[0, 1] = w2
    model.linear1.bias[0] = b
model.linear1.weight, model.linear1.bias

## Compile the Model

In [ ]:
# serious projects need hardware-specific compile and load
# compiled_model = torch.compile(model, backend=...)
# we go for uncompiled to be hardware agnostic
compiled_model = model

## Train the Model

In [ ]:
for epoch in range(num_epochs):
    if (epoch+1) % 1 == 0:
        print('epoch:', epoch+1)

    epoch_loss = 0
    for batch_idx, batch in enumerate(data_train_loader, 1):
        X, Y = batch[0], batch[1]
        Y_pred = compiled_model(X)
        loss = loss_fcn(Y_pred, Y[:, None])
        epoch_loss += loss.item() * X.shape[0]
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    if (epoch+1) % 1 == 0:
        with torch.no_grad():
            loss_test = loss_fcn(model(data_test[:][0]),
                                 data_test[:][1][:, None])
        print('empirical risk train:',
              '%0.15e' % (epoch_loss / X_train.shape[0]),
              ', empirical risk test:',
              '%0.15e' % loss_test.item())

## Check Model Parameters

In [ ]:
tmp = model.linear1.weight.detach().numpy()
print('%+0.15e' % tmp[0, 0], '\n%+0.15e' % tmp[0, 1])
tmp = model.linear1.bias.detach().numpy()
print('%+0.15e' % tmp[0])

## Model Test

### Empirical Risk

In [ ]:
with torch.no_grad():
    loss_train = loss_fcn(
        model(data_train[:][0]),  data_train[:][1][:, None])
    print('empirical risk train:', '%0.15e' % loss_train.item())
    loss_train = loss_fcn(
        model(data_test[:][0]),  data_test[:][1][:, None])
    print('empirical risk test: ', '%0.15e' % loss_test.item())

### Prep for Class Prediction Metrics

In [ ]:
# from here, we work with numpy rank 1 arrays, i.e. (8000,) and (2000,)
y_true_train = data_train[:][1][:, None][:, 0].numpy()
y_pred_train = model.predict_class(data_train[:][0])[:, 0].numpy()
y_true_test = data_test[:][1][:, None][:, 0].numpy()
y_pred_test = model.predict_class(data_test[:][0])[:, 0].numpy()

### Confusion Matrix

In [ ]:
print('confusion matrix train absolute')
print(confusion_matrix(
    y_true_train,
    y_pred_train,
    normalize=None))
print('confusion matrix train in %')
print(confusion_matrix(
    y_true_train,
    y_pred_train,
    normalize='all')*100)
print('\nconfusion matrix test absolute')
print(confusion_matrix(
    y_true_test,
    y_pred_test,
    normalize=None))
print('confusion matrix test in %')
print(confusion_matrix(
    y_true_test,
    y_pred_test,
    normalize='all')*100)

### Precision, Recall, F1Score, Support

In [ ]:
p, r, f, s = precision_recall_fscore_support(
    y_true_train, y_pred_train)
print(p, r, f, s)
p, r, f, s = precision_recall_fscore_support(
    y_true_test, y_pred_test)
print(p, r, f, s)

### Accuracy, Balanced Accuracy

We have a very balanced data set, hence both values are very similar

In [ ]:
a = accuracy_score(
    y_true_train, y_pred_train)
ba = balanced_accuracy_score(
    y_true_train, y_pred_train)
print(a, ba)

a = accuracy_score(
    y_true_test, y_pred_test)
ba = balanced_accuracy_score(
    y_true_test, y_pred_test)
print(a, ba)

## Plot Data Points and Decision Plane

In [ ]:
# get model parameters
w = model.linear1.weight.detach().numpy()
b = model.linear1.bias.detach().numpy()

# get probabilities in the prediction plane
levels = [0.0, 0.05, 0.1, 0.37, 0.5, 0.63, 0.9, 0.95, 1]
f1, f2 = np.arange(-5, 5, 0.05), np.arange(-5, 5, 0.05)
xv, yv = np.meshgrid(f1, f2)
# the model prediction as manual one-liner, this yields a probability
prob_plane = my_sigmoid(w[0, 0] * xv + w[0, 1] * yv + b[0])
# hard decision boundary for classes 0,1:
# prob_plane = predict_class(prob_plane)

plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(X_train[Y_train == 1, 0],
         X_train[Y_train == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training class '1' " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 2)
plt.plot(X_train[Y_train == 0, 0],
         X_train[Y_train == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training class '0' " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 3)
plt.plot(X_test[Y_test == 1, 0],
         X_test[Y_test == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test class '1' " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 4)
plt.plot(X_test[Y_test == 0, 0],
         X_test[Y_test == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test class '0' " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.